# Greedy Algorithms — Beginner's Guide

> A structured notebook covering theory, patterns, practice problems, and LeetCode solutions.

---

## Table of Contents
1. [Core Concept](#core)
2. [Two Key Properties](#properties)
3. [Greedy vs Dynamic Programming](#gvsdp)
4. [Classic Patterns](#patterns)
5. [LeetCode Problems (Easy → Hard)](#leetcode)
   - 455 — Assign Cookies
   - 860 — Lemonade Change
   - 55 — Jump Game
   - 45 — Jump Game II
   - 435 — Non-overlapping Intervals
   - 452 — Minimum Arrows to Burst Balloons
   - 406 — Queue Reconstruction by Height
   - 135 — Candy
6. [When Greedy Fails — Counter Examples](#fails)
7. [Practice Questions](#practice)

---
## 1. Core Concept

A **greedy algorithm** makes the *locally optimal choice* at each step, hoping it leads to a globally optimal solution.

- No backtracking
- No reconsideration of past choices
- Just: **"what looks best right now?"**

### The key question to ask before using greedy:
> *Does choosing the best option at each step guarantee the best overall answer?*

If **yes** → greedy works.  
If **no** → use DP or backtracking.

---
## 2. Two Key Properties

For greedy to produce a correct solution, the problem must have both:

### Greedy Choice Property
A globally optimal solution can be reached by making locally optimal (greedy) choices. You never need to "undo" a greedy pick.

### Optimal Substructure
An optimal solution to the problem contains optimal solutions to its subproblems. After making a greedy choice, the remaining problem is the same type and can be solved greedily too.

> Note: DP also requires optimal substructure — the difference is greedy commits to one choice, while DP explores all choices.

---
## 3. Greedy vs Dynamic Programming

| | Greedy | Dynamic Programming |
|---|---|---|
| **Choices** | One per step, never revisited | All sub-possibilities explored |
| **Time complexity** | O(n log n) or better | O(n²) or more |
| **Space** | O(1) or O(n) | Usually O(n) or O(n²) |
| **When correct** | Only when greedy choice property holds | Always (if formulated correctly) |
| **Examples** | Activity selection, Dijkstra | 0/1 knapsack, coin change (arbitrary) |

### Quick heuristic
If the problem involves **"minimum/maximum number of X"** and you can **sort elements by a single key** (end time, ratio, size) → try greedy first.  
If your greedy fails a counter-example you construct → switch to DP.

---
## 4. Classic Greedy Patterns

| Problem | Greedy? | Strategy |
|---|---|---|
| Coin change (US coins: 1,5,10,25) | ✅ Greedy | Always take largest coin ≤ remaining |
| Coin change (arbitrary denominations) | ❌ DP needed | Greedy choice can block better combos |
| Fractional knapsack | ✅ Greedy | Sort by value/weight ratio, take fractions |
| 0/1 Knapsack | ❌ DP needed | Must consider all combinations of whole items |
| Activity selection / intervals | ✅ Greedy | Sort by end time, pick earliest ending |
| Longest increasing subsequence | ❌ DP needed | Need to track all possible endings |
| Dijkstra's shortest path | ✅ Greedy | Always expand nearest unvisited node |
| Shortest path with negative weights | ❌ Bellman-Ford | Greedy breaks with negative edges |

---
## 5. LeetCode Problems

### 🟢 455 — Assign Cookies (Easy)
**Pattern**: Sort + Two Pointers

**Problem**: Kids have greed factors `g[]`, cookies have sizes `s[]`. Assign at most one cookie per child. A cookie satisfies a child only if `size >= greed`. Maximize number of satisfied children.

**Greedy insight**: Satisfy the least greedy child with the smallest sufficient cookie. This wastes the least resources.

**Steps**:
1. Sort both arrays
2. Use two pointers — try to match current cookie to current child
3. If cookie satisfies child → move both pointers
4. Else → move only cookie pointer (try a bigger cookie)

In [ ]:
def assign_cookies(g, s):
    """
    LeetCode 455 — Assign Cookies
    Time: O(n log n + m log m)  Space: O(1)
    """
    g.sort()
    s.sort()
    child = cookie = 0

    while child < len(g) and cookie < len(s):
        if s[cookie] >= g[child]:  # cookie satisfies this child
            child += 1
        cookie += 1  # always move to next cookie

    return child


# Test cases
print(assign_cookies([1, 2, 3], [1, 1]))        # Expected: 1
print(assign_cookies([1, 2], [1, 2, 3]))         # Expected: 2
print(assign_cookies([10, 9, 8, 7], [5, 6, 7])) # Expected: 1

### 🟢 860 — Lemonade Change (Easy)
**Pattern**: Simulation / Greedy Cash Register

**Problem**: Customers pay with $5, $10, or $20 bills. Lemonade costs $5. Can you always give correct change?

**Greedy insight**: When giving change for $20, prefer to give a $10 bill before two $5 bills. $5 bills are more versatile (needed for $10 change too), so preserve them.

In [ ]:
def lemonade_change(bills):
    """
    LeetCode 860 — Lemonade Change
    Time: O(n)  Space: O(1)
    """
    five = ten = 0

    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            if five == 0:
                return False
            five -= 1
            ten += 1
        else:  # bill == 20
            if ten > 0 and five > 0:   # greedy: prefer $10 + $5
                ten -= 1
                five -= 1
            elif five >= 3:             # fallback: three $5 bills
                five -= 3
            else:
                return False

    return True


# Test cases
print(lemonade_change([5, 5, 5, 10, 20]))        # Expected: True
print(lemonade_change([5, 5, 10, 10, 20]))        # Expected: False
print(lemonade_change([5, 5, 5, 5, 20, 20, 5]))  # Expected: True

### 🟡 55 — Jump Game (Medium)
**Pattern**: Track max_reach

**Problem**: `nums[i]` = max jump length from index `i`. Can you reach the last index?

**Greedy insight**: At each step, update the furthest index reachable. If you ever land at an index beyond your current reach → stuck.

**Key variable**: `max_reach = max(max_reach, i + nums[i])`

In [ ]:
def can_jump(nums):
    """
    LeetCode 55 — Jump Game
    Time: O(n)  Space: O(1)
    """
    max_reach = 0

    for i, jump in enumerate(nums):
        if i > max_reach:          # can't reach this index
            return False
        max_reach = max(max_reach, i + jump)

    return True


# Test cases
print(can_jump([2, 3, 1, 1, 4]))  # Expected: True
print(can_jump([3, 2, 1, 0, 4]))  # Expected: False
print(can_jump([0]))              # Expected: True (already at end)
print(can_jump([2, 0, 0]))        # Expected: True

### 🟡 45 — Jump Game II (Medium)
**Pattern**: BFS-level greedy

**Problem**: Same setup as above. Find the **minimum number of jumps** to reach the last index.

**Greedy insight**: Think of it like BFS levels. Track the end of the current "jump range" and the farthest you can reach from within that range. When you exhaust the current range, you must make a jump.

In [ ]:
def jump(nums):
    """
    LeetCode 45 — Jump Game II
    Time: O(n)  Space: O(1)
    """
    jumps = 0
    current_end = 0   # end of current jump's reach
    farthest = 0      # farthest index reachable from current level

    for i in range(len(nums) - 1):  # don't need to jump from last index
        farthest = max(farthest, i + nums[i])
        if i == current_end:         # exhausted current jump range
            jumps += 1
            current_end = farthest   # expand to farthest reachable

    return jumps


# Test cases
print(jump([2, 3, 1, 1, 4]))     # Expected: 2 (0->1->4)
print(jump([2, 3, 0, 1, 4]))     # Expected: 2
print(jump([1, 2, 1, 1, 1]))     # Expected: 3
print(jump([1]))                 # Expected: 0

### 🟡 435 — Non-overlapping Intervals (Medium)
**Pattern**: Sort by end time (activity selection)

**Problem**: Given intervals, find the minimum number to **remove** so no two overlap.

**Greedy insight**: This is the flip side of activity selection — maximize the intervals you *keep* (non-overlapping), then count the ones removed.

Sort by end time. Greedily keep each interval that starts at or after the previous kept interval's end.

In [ ]:
def erase_overlap_intervals(intervals):
    """
    LeetCode 435 — Non-overlapping Intervals
    Time: O(n log n)  Space: O(1)
    """
    if not intervals:
        return 0

    intervals.sort(key=lambda x: x[1])  # sort by end time
    removed = 0
    prev_end = intervals[0][1]

    for start, end in intervals[1:]:
        if start < prev_end:  # overlap detected
            removed += 1      # remove current interval (it ends later)
        else:
            prev_end = end    # keep this interval, update boundary

    return removed


# Test cases
print(erase_overlap_intervals([[1,2],[2,3],[3,4],[1,3]]))  # Expected: 1
print(erase_overlap_intervals([[1,2],[1,2],[1,2]]))         # Expected: 2
print(erase_overlap_intervals([[1,2],[2,3]]))               # Expected: 0

### 🟡 452 — Minimum Arrows to Burst Balloons (Medium)
**Pattern**: Sort by end time (interval grouping)

**Problem**: Balloons span `[x_start, x_end]` on a 2D wall. Arrows shot vertically burst all balloons the arrow passes through. Find minimum arrows needed to burst all balloons.

**Greedy insight**: Same core pattern as 435. An arrow placed at the end of the earliest-ending balloon bursts as many overlapping balloons as possible.

In [ ]:
def find_min_arrow_shots(points):
    """
    LeetCode 452 — Minimum Arrows to Burst Balloons
    Time: O(n log n)  Space: O(1)
    """
    if not points:
        return 0

    points.sort(key=lambda x: x[1])  # sort by end position
    arrows = 1
    arrow_pos = points[0][1]         # place first arrow at end of first balloon

    for start, end in points[1:]:
        if start > arrow_pos:        # current balloon not hit by last arrow
            arrows += 1
            arrow_pos = end          # place new arrow at end of this balloon

    return arrows


# Test cases
print(find_min_arrow_shots([[10,16],[2,8],[1,6],[7,12]]))  # Expected: 2
print(find_min_arrow_shots([[1,2],[3,4],[5,6],[7,8]]))     # Expected: 4
print(find_min_arrow_shots([[1,2],[2,3],[3,4],[4,5]]))     # Expected: 2

### 🟡 406 — Queue Reconstruction by Height (Medium)
**Pattern**: Sort + Insert

**Problem**: People described as `[h, k]` where `h` = height, `k` = number of people in front with height ≥ h. Reconstruct the queue.

**Greedy insight**:
1. Sort by height descending (tallest first), break ties by `k` ascending
2. Insert each person at index `k` in the result

Since taller people are placed first, inserting a shorter person at index `k` only counts taller people — exactly the `k` constraint.

In [ ]:
def reconstruct_queue(people):
    """
    LeetCode 406 — Queue Reconstruction by Height
    Time: O(n^2)  Space: O(n)
    """
    # Sort: tallest first, ties broken by k ascending
    people.sort(key=lambda x: (-x[0], x[1]))

    result = []
    for person in people:
        result.insert(person[1], person)  # insert at index k

    return result


# Test cases
people = [[7,0],[4,4],[7,1],[5,0],[6,1],[5,2]]
print(reconstruct_queue(people))
# Expected: [[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]]

people2 = [[6,0],[5,0],[4,0],[3,2],[2,2],[1,4]]
print(reconstruct_queue(people2))
# Expected: [[4,0],[5,0],[2,2],[3,2],[1,4],[6,0]]

### 🔴 135 — Candy (Hard)
**Pattern**: Two-pass greedy

**Problem**: Children in a line, each with a rating. Rules:
- Every child gets ≥ 1 candy
- Children with higher rating than a *neighbor* get more candy than that neighbor

Minimize total candies.

**Greedy insight**: One greedy pass can't satisfy both left and right neighbor constraints simultaneously. Use two passes — each enforcing one direction — then take the max at each position.

- **Pass 1 (L→R)**: If `ratings[i] > ratings[i-1]` → `candy[i] = candy[i-1] + 1`
- **Pass 2 (R→L)**: If `ratings[i] > ratings[i+1]` → `candy[i] = max(candy[i], candy[i+1] + 1)`

In [ ]:
def candy(ratings):
    """
    LeetCode 135 — Candy
    Time: O(n)  Space: O(n)
    """
    n = len(ratings)
    candies = [1] * n  # everyone starts with 1

    # Pass 1: left to right — enforce left neighbor constraint
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            candies[i] = candies[i - 1] + 1

    # Pass 2: right to left — enforce right neighbor constraint
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            candies[i] = max(candies[i], candies[i + 1] + 1)

    return sum(candies)


# Test cases
print(candy([1, 0, 2]))        # Expected: 5  → [2,1,2]
print(candy([1, 2, 2]))        # Expected: 4  → [1,2,1]
print(candy([1, 3, 2, 2, 1])) # Expected: 7  → [1,3,1,2,1]
print(candy([1, 2, 87, 87, 87, 2, 1]))  # Expected: 13

---
## 6. When Greedy Fails — Counter Examples

In [ ]:
# Counter-example 1: Coin change with arbitrary denominations
# Greedy: always pick the largest coin <= remaining

def coin_change_greedy(coins, amount):
    """WRONG for arbitrary denominations"""
    coins.sort(reverse=True)
    count = 0
    for coin in coins:
        while amount >= coin:
            amount -= coin
            count += 1
    return count if amount == 0 else -1


def coin_change_dp(coins, amount):
    """CORRECT — dynamic programming"""
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for a in range(1, amount + 1):
        for coin in coins:
            if coin <= a:
                dp[a] = min(dp[a], dp[a - coin] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1


coins = [1, 3, 4]
amount = 6

print(f"Greedy: {coin_change_greedy(coins[:], amount)} coins")  # 3 coins: 4+1+1 (WRONG)
print(f"DP:     {coin_change_dp(coins, amount)} coins")          # 2 coins: 3+3 (CORRECT)
print()
print("Moral: Greedy only works for canonical coin systems (e.g. US coins)")

In [ ]:
# Counter-example 2: 0/1 Knapsack — greedy by value/weight ratio fails

def knapsack_greedy(weights, values, capacity):
    """WRONG for 0/1 knapsack"""
    items = sorted(zip(weights, values), key=lambda x: x[1]/x[0], reverse=True)
    total_value = 0
    for w, v in items:
        if capacity >= w:
            capacity -= w
            total_value += v
    return total_value


def knapsack_dp(weights, values, capacity):
    """CORRECT — dynamic programming"""
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i-1][w]
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i][w], dp[i-1][w - weights[i-1]] + values[i-1])
    return dp[n][capacity]


weights = [3, 4, 5]
values  = [4, 5, 6]
capacity = 7

# Ratios: 4/3=1.33, 5/4=1.25, 6/5=1.2 → greedy picks item 0 (w=3,v=4) then item 1 (w=4,v=5) → total=9
# Optimal: item 1 (w=4,v=5) + nothing? No — item 0+2 = w=8 > 7. Item 1+0 = w=7, v=9. Same here actually!
# Try a case where greedy truly fails:
weights2 = [1, 3, 4, 5]
values2  = [1, 4, 5, 7]
capacity2 = 7

print(f"Greedy: {knapsack_greedy(weights2[:], values2[:], capacity2)}")  # May pick suboptimally
print(f"DP:     {knapsack_dp(weights2, values2, capacity2)}")             # Always optimal

---
## 7. Practice Questions

Try these before looking at the solutions above!

### Conceptual
1. What is the greedy algorithm strategy in one sentence?
2. What two properties must a problem have for greedy to work?
3. What is the key difference between greedy and dynamic programming?
4. Why does greedy fail for 0/1 knapsack but work for fractional knapsack?
5. Prove or disprove: greedy always works for coin change.

### Pattern Recognition — Greedy or DP?
For each, state whether greedy works and why:
1. Find the minimum number of intervals to remove so none overlap
2. Find the minimum number of coins to make an amount (coins = [1, 3, 4])
3. Find the longest increasing subsequence
4. Find the shortest path in a graph with all non-negative weights
5. Assign tasks to minimize total completion time (each task has one duration)

### Coding Challenges (no hints!)
1. **Gas Station** (LeetCode 134) — greedy track of cumulative surplus
2. **Partition Labels** (LeetCode 763) — greedy interval merging
3. **Task Scheduler** (LeetCode 621) — greedy frequency-based scheduling
4. **Hand of Straights** (LeetCode 846) — greedy grouping by smallest element

In [ ]:
# Your sandbox — write your solutions here!

# Example: Gas Station (LeetCode 134)
def can_complete_circuit(gas, cost):
    # TODO: implement
    pass


# Example: Partition Labels (LeetCode 763)
def partition_labels(s):
    # TODO: implement
    pass


# Test your solutions below


---
## Summary Cheatsheet

```
GREEDY DECISION TREE
====================
Problem asks for min/max of something?
  ├── Can you sort elements by a single key and pick greedily?
  │     ├── YES → Try greedy. Verify with a counter-example.
  │     └── NO  → Likely DP or backtracking
  └── Does your greedy fail a counter-example you construct?
        └── YES → Switch to DP

COMMON GREEDY KEYS
==================
Intervals    → Sort by end time
Knapsack     → Sort by value/weight ratio (fractional only!)
Huffman      → Always merge two smallest frequencies
Dijkstra     → Always expand nearest unvisited node
Jump Game    → Track max_reach
Candy        → Two passes (one per constraint direction)
```